In [1]:
%pip install -q mediapipe

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install mediapipe opencv-python

Note: you may need to restart the kernel to use updated packages.


In [4]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time
import math

# ================== MODEL PATHS ==================
POSE_MODEL_PATH = "pose_landmarker_full.task"
HAND_MODEL_PATH = "hand_landmarker.task"

# ================== BASE OPTIONS ==================
BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode

# ================== POSE SETUP ==================
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions

pose_options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1
)

# ================== HAND SETUP ==================
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions

hand_options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=1
)

# ================== CALCULATE DISTANCE ==================
def distance(p1, p2):
    return math.sqrt((p1.x - p2.x) ** 2 + (p1.y - p2.y) ** 2)

# ================== CAMERA ==================
cap = cv2.VideoCapture(0)

window_name = "Physiotherapy Hand Check"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

with PoseLandmarker.create_from_options(pose_options) as pose_landmarker, \
     HandLandmarker.create_from_options(hand_options) as hand_landmarker:

    while True:

        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        timestamp_ms = int(time.time() * 1000)

        # ================== HAND DETECTION ==================
        hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        if hand_result.hand_landmarks:

            for hand_landmarks in hand_result.hand_landmarks:

                # رسم النقاط
                for lm in hand_landmarks:
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    cv2.circle(frame, (cx, cy), 4, (255, 0, 0), -1)

                # نحسب المسافة بين أطراف الأصابع والراحة
                palm = hand_landmarks[0]

                fingertips = [8, 12, 16, 20]

                total_distance = 0
                for tip in fingertips:
                    total_distance += distance(hand_landmarks[tip], palm)

                avg_distance = total_distance / 4

                # ================== تحديد حالة القبضة ==================
                if avg_distance < 0.18:
                    status = "CLOSED (Correct)"
                    color = (0, 255, 0)
                elif avg_distance < 0.25:
                    status = "Half Closed"
                    color = (0, 255, 255)
                else:
                    status = "OPEN"
                    color = (0, 0, 255)

                # عرض النتيجة
                cv2.putText(frame, status,
                            (30, 60),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1, color, 3)

        # ================== DISPLAY ==================
        cv2.imshow(window_name, frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

# ================== CLEANUP ==================
cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 